<a href="https://colab.research.google.com/github/zamanuddinkhan/Python-AI-LLM/blob/main/Messages_ToolCalling_Reducers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Chain Using LangGraph
In this section we will see how we can build a simple chain using Langgraph that uses 4 important concepts

- How to use chat messages as our graph state
- How to use chat models in graph nodes
- How to bind tools to our LLM in chat models
- How to execute the tools call in our graph nodes

In [ ]:

from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"]=userdata.get('OPENAI_API_KEY')
os.environ["GROQ_API_KEY"]=userdata.get('GROQ_API_KEY')


#### How to use chat messages as our graph state
##### Messages

We can use messages which can be used to capture different roles within a conversation.
LangChain has various message types including HumanMessage, AIMessage, SystemMessage and ToolMessage.
These represent a message from the user, from chat model, for the chat model to instruct behavior, and from a tool call.

Every message have these important components.

- content - content of the message
- name - Specify the name of author
- response_metadata - optionally, a dict of metadata (e.g., often populated by model provider for AIMessages)



In [ ]:
from langchain_core.messages import AIMessage,HumanMessage
from pprint import pprint

messages=[AIMessage(content=f"Please tell me how can I help",name="LLMModel")]
messages.append(HumanMessage(content=f"I want to learn coding",name="Ajit"))
messages.append(AIMessage(content=f"Which programming language you want to learn",name="LLMModel"))
messages.append(HumanMessage(content=f"I want to learn python programming language",name="Ajit"))

for message in messages:
    message.pretty_print()



================================== Ai Message ==================================
Name: LLMModel

Please tell me how can I help
================================ Human Message =================================
Name: Ajit

I want to learn coding
================================== Ai Message ==================================
Name: LLMModel

Which programming language you want to learn
================================ Human Message =================================
Name: Ajit

I want to learn python programming language


### Chat Models

We can use the sequence of message as input with the chatmodels using LLM's and OPENAI.

In [ ]:
!pip install langchain_groq langchain_openai langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 22.6 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1


In [ ]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="llama-3.1-8b-instant")
result=llm.invoke(messages)

In [ ]:
result.content

'Python is a popular and versatile language, widely used in various fields such as web development, data science, machine learning, and more. Here\'s a step-by-step guide to help you get started:\n\n**1. Install Python:**\nDownload and install the latest version of Python from the official Python website: <https://www.python.org/downloads/>\n\n**2. Choose a Text Editor or IDE:**\nA text editor or IDE (Integrated Development Environment) is where you\'ll write your code. Some popular choices for Python include:\n * PyCharm (free community edition)\n * Visual Studio Code (free)\n * Sublime Text (free trial, paid version)\n * Notepad++ (free)\n\n**3. Learn the Basics:**\nStart with basic tutorials and online courses to understand the syntax and data types in Python. Some resources include:\n * Codecademy\'s Python Course (interactive coding environment)\n * Python.org (official tutorials and documentation)\n * W3Schools\' Python Tutorial (concise and easy to follow)\n\n**4. Practice with 

In [ ]:
import IPython.display as display
display.Markdown(result.content)

Python is a popular and versatile language used in many areas such as web development, data analysis, artificial intelligence, and more. Here's a step-by-step guide to help you get started:

**Step 1: Learn the Basics**

1. **Get familiar with the syntax**: Python's syntax is easy to read and write.
2. **Understand variables, data types, and operators**: Learn how to store and manipulate data in Python.
3. **Basic control structures**: Understand how to use if-else statements, for loops, and while loops.

**Step 2: Practice with Tutorials and Exercises**

1. **Codecademy's Python Course**: A interactive and comprehensive course that covers the basics and beyond.
2. **Python.org**: The official Python website has a tutorial for beginners.
3. **W3Schools' Python Tutorial**: A concise and easy-to-follow tutorial.

**Step 3: Work on Projects**

1. **Build a simple calculator**: Create a calculator that takes user input and performs basic arithmetic operations.
2. **Play with data**: Use libraries like Pandas and NumPy to work with data and perform analysis.
3. **Build a game**: Create a simple game like Tic-Tac-Toe or Snake.

**Step 4: Learn Advanced Concepts**

1. **Object-Oriented Programming**: Learn how to create classes and objects in Python.
2. **File Input/Output**: Understand how to read and write files in Python.
3. **Error Handling**: Learn how to handle errors and exceptions in Python.

**Step 5: Join a Community**

1. **Reddit's r/learnpython**: A community of Python learners and experts.
2. **Stack Overflow**: A Q&A platform for programmers.
3. **Python Subreddit**: A community of Python enthusiasts.

**Tools and Resources**

1. **PyCharm**: A popular IDE for Python development.
2. **Visual Studio Code**: A lightweight and versatile code editor.
3. **Python Virtual Environments**: Learn how to create and manage virtual environments.

Remember, practice is key to learning Python. Start with the basics, build projects, and gradually move on to advanced concepts. Good luck!

Which step do you want to start with?

### Tools
Tools can be integrated with the LLM models to interact with external systems. External systems can be API's, third party tools.

Whenever a query is asked the model can choose to call the tool and this query is based on the
natural language input and this will return an output that matches the tool's schema

In [ ]:
#Custom Tool1

def add(a:int,b:int)-> int:
    """ Add a and b
    Args:
        a (int): first int
        b (int): second int

    Returns:
        int
    """
    return a+b

In [ ]:
#Custom Tool2
def multiply(a:int,b:int)-> int:
    """ Multiply a and b
    Args:
        a (int): first int
        b (int): second int

    Returns:
        int
    """
    return a*b

In [ ]:
llm

In [ ]:
### Binding tool with llm

llm_with_tools=llm.bind_tools([add,multiply])

tool_call=llm_with_tools.invoke([HumanMessage(content=f"What is 2 plus 2",name="Ajit")])

In [ ]:
tool_call.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  add (jd1dc4x03)
 Call ID: jd1dc4x03
  Args:
    a: 2
    b: 2


In [ ]:
tool_call.tool_calls

[{'name': 'add',
  'args': {'a': 2, 'b': 2},
  'id': 'jd1dc4x03',
  'type': 'tool_call'}]

In [ ]:
tool_call=llm_with_tools.invoke([HumanMessage(content=f"What is 45 multiply 32",name="Ajit")])

In [ ]:
tool_call.pretty_print()

================================== Ai Message ==================================

<multiply a=45 b=32>


### Using messages as state

In [ ]:
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage

class State(TypedDict):
    message:list[AnyMessage]

#### Reducers
Now, we have a minor problem!

As we discussed, each node will return a new value for our state key messages.

But, this new value will override the prior messages value.

As our graph runs, we want to append messages to our messages state key.

We can use reducer functions to address this.

Reducers allow us to specify how state updates are performed.

If no reducer function is specified, then it is assumed that updates to the key should override it as we saw before.

But, to append messages, we can use the pre-built add_messages reducer.

This ensures that any messages are appended to the existing list of messages.

We simply need to annotate our messages key with the add_messages reducer function as metadata.

In [ ]:
from langgraph.graph.message import add_messages
from typing import Annotated
class State(TypedDict):
    messages:Annotated[list[AnyMessage],add_messages]

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


NameError: name 'TypedDict' is not defined

### Reducers with add_messages

In [ ]:
initial_messages=[AIMessage(content=f"Please tell me how can I help",name="LLMModel")]
initial_messages.append(HumanMessage(content=f"I want to learn coding",name="Ajit"))
initial_messages

[AIMessage(content='Please tell me how can I help', additional_kwargs={}, response_metadata={}, name='LLMModel', tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I want to learn coding', additional_kwargs={}, response_metadata={}, name='Ajit')]

In [ ]:
ai_message=AIMessage(content=f"Which programming language you want to learn",name="LLMModel")
ai_message

AIMessage(content='Which programming language you want to learn', additional_kwargs={}, response_metadata={}, name='LLMModel', tool_calls=[], invalid_tool_calls=[])

In [ ]:
### Reducers add_messages is to append instead of override
add_messages(initial_messages,ai_message)

NameError: name 'add_messages' is not defined

In [ ]:
## chatbot node functionality
def llm_tool(state:State):
    return {"messages":[llm_with_tools.invoke(state["messages"])]}

NameError: name 'State' is not defined

In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
builder=StateGraph(State)

builder.add_node("llm_tool",llm_tool)

builder.add_edge(START,"llm_tool")
builder.add_edge("llm_tool",END)

graph=builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
## invocation

messages=graph.invoke({"messages":"What is 2 plus 2"})

for message in messages["messages"]:
    message.pretty_print()

In [ ]:
tools=[add,multiply]

Adding tool node

In [ ]:
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition  #conditional edge


builder=StateGraph(State)

## Add nodes

builder.add_node("llm_tool",llm_tool)
builder.add_node("tools",ToolNode(tools))

## Add Edge
builder.add_edge(START,"llm_tool")
builder.add_conditional_edges(
    "llm_tool",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
    # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
    tools_condition
)
builder.add_edge("tools",END)


graph_builder = builder.compile()



In [ ]:
display(Image(graph_builder.get_graph().draw_mermaid_png()))

In [ ]:
## invocation

messages=graph_builder.invoke({"messages":"What is 2 plus 2"})

for message in messages["messages"]:
    message.pretty_print()

In [ ]:
messages=graph_builder.invoke({"messages":"What is 45 times 56"})

for message in messages["messages"]:
    message.pretty_print()

In [ ]:
messages=graph.invoke({"messages":"Explain Machine Learning in simple terms"})

for message in messages["messages"]:
    message.pretty_print()